In [1]:
from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Connecting to the database in Google BigQuery (DA dataset)

In [2]:
auth.authenticate_user()
client = bigquery.Client(project="data-analytics-mate")

query = """
with session_info as (
  select
  s.date,
  s.ga_session_id,
  sp.country,
  sp.device,
  sp.continent,
  sp.channel,
  ab.test,
  ab.test_group
  from `data-analytics-mate.DA.ab_test`ab
  join `data-analytics-mate.DA.session`s
  on ab.ga_session_id = s.ga_session_id
  join `data-analytics-mate.DA.session_params`sp
  on sp.ga_session_id = ab.ga_session_id




),
sessions_with_orders as
(select
  session_info.date,
  session_info.country,
  session_info.device,
  session_info.continent,
  session_info.channel,
  session_info.test,
  session_info.test_group,
  count(distinct o.ga_session_id) as session_with_order

from `data-analytics-mate.DA.order`o
join session_info
on session_info.ga_session_id = o.ga_session_id
group by
  session_info.date,
  session_info.country,
  session_info.device,
  session_info.continent,
  session_info.channel,
  session_info.test,
  session_info.test_group
),
events as
(select
  session_info.date,
  session_info.country,
  session_info.device,
  session_info.continent,
  session_info.channel,
  session_info.test,
  session_info.test_group,
  ep.event_name,
  count(ep.ga_session_id) as event_cnt
from `data-analytics-mate.DA.event_params`ep
join session_info
on session_info.ga_session_id = ep.ga_session_id
group by
  session_info.date,
  session_info.country,
  session_info.device,
  session_info.continent,
  session_info.channel,
  session_info.test,
  session_info.test_group,
  ep.event_name),

  sessions as

  (select
  session_info.date,
  session_info.country,
  session_info.device,
  session_info.continent,
  session_info.channel,
  session_info.test,
  session_info.test_group,
   count(distinct session_info.ga_session_id) as session_cnt
   from session_info
   group by
  session_info.date,
  session_info.country,
  session_info.device,
  session_info.continent,
  session_info.channel,
  session_info.test,
  session_info.test_group
  ),




  accounts as (

  select session_info.date,
  session_info.country,
  session_info.device,
  session_info.continent,
  session_info.channel,
  session_info.test,
  session_info.test_group,
  count(distinct acs.ga_session_id) as new_account_cnt
  from `data-analytics-mate.DA.account_session`acs
  join session_info
  on session_info.ga_session_id = acs.ga_session_id
  group by
  session_info.date,
  session_info.country,
  session_info.device,
  session_info.continent,
  session_info.channel,
  session_info.test,
  session_info.test_group)




  select




  sessions_with_orders.date,
  sessions_with_orders.country,
  sessions_with_orders.device,
  sessions_with_orders.continent,
  sessions_with_orders.channel,
  sessions_with_orders.test,
  sessions_with_orders.test_group,
  'sessions_with_orders' as event_name,
  sessions_with_orders.session_with_order as value
  from sessions_with_orders




  union all
  select
  events.date,
  events.country,
  events.device,
  events.continent,
  events.channel,
  events.test,
  events.test_group,
  event_name,
  event_cnt as value
  from events




union all

select
  sessions.date,
  sessions.country,
  sessions.device,
  sessions.continent,
  sessions.channel,
  sessions.test,
  sessions.test_group,
  'sessions' as event_name,
  sessions.session_cnt as value
  from sessions




union all
select
accounts.date,
accounts.country,
accounts.device,
accounts.continent,
accounts.channel,
accounts.test,
accounts.test_group,
'new account' as event_name,
 new_account_cnt as value
 from accounts

 """
query_job = client.query(query)
results = query_job.result()

#df = results.to_dataframe()
rows = [dict(row) for row in results]
df = pd.DataFrame(rows)
df.head()



,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-06,Slovakia,mobile,Europe,Paid Search,2,2,sessions_with_orders,1
1,2020-11-06,Slovakia,mobile,Europe,Paid Search,1,2,sessions_with_orders,1
2,2020-12-09,El Salvador,mobile,Americas,Direct,4,2,sessions_with_orders,1
3,2020-12-09,El Salvador,mobile,Americas,Direct,3,2,sessions_with_orders,1
4,2020-12-21,Slovakia,mobile,Europe,Organic Search,4,2,sessions_with_orders,1


In [3]:
from google.colab import files
df.to_csv("df.csv", index=False)
files.download("df.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   date        800996 non-null  object
 1   country     800996 non-null  object
 2   device      800996 non-null  object
 3   continent   800996 non-null  object
 4   channel     800996 non-null  object
 5   test        800996 non-null  int64 
 6   test_group  800996 non-null  int64 
 7   event_name  800996 non-null  object
 8   value       800996 non-null  int64 
dtypes: int64(3), object(6)
memory usage: 55.0+ MB


# Statistical Significance Testing

In [5]:
# Select required events (metrics)
df_metrix = df[df["event_name"].isin(["add_payment_info", "add_shipping_info", "begin_checkout", "new account"])]
df_metrix_grouped = (df_metrix.groupby(["test", "test_group", "event_name"])["value"].sum().reset_index())

# Count the number of sessions (this will be the denominator)
df_sessions = df[df['event_name'] == "sessions"]
df_sessions_grouped = (df_sessions.groupby(["test", "test_group"])["value"].sum().reset_index().rename(columns={"value": "sessions"}))

# Main calculation loop
results = []

for test_number in df_metrix_grouped["test"].unique():
    # Select data for the current test only
    df_test = df_metrix_grouped[df_metrix_grouped["test"] == test_number]

    # Iterate over each event (metric)
    for event_name in df_test["event_name"].unique():
        # Data for control group (1)
        group_1 = df_test[(df_test["test_group"] == 1) & (df_test["event_name"] == event_name)]
        # Data for test group (2)
        group_2 = df_test[(df_test["test_group"] == 2) & (df_test["event_name"] == event_name)]

        # Number of users who completed the event
        group1 = group_1["value"].sum()
        group2 = group_2["value"].sum()

        # Get session counts for each group from df_sessions_grouped
        div1 = df_sessions_grouped.loc[(df_sessions_grouped["test"] == test_number) & (df_sessions_grouped["test_group"] == 1), "sessions"].sum()
        div2 = df_sessions_grouped.loc[(df_sessions_grouped["test"] == test_number) & (df_sessions_grouped["test_group"] == 2), "sessions"].sum()

        # Calculate conversion rate (share of users who completed the event)
        conversion_1 = group1 / div1 if div1 > 0 else 0
        conversion_2 = group2 / div2 if div2 > 0 else 0

        # Perform z-test to compare proportions between groups
        if div1 > 0 and div2 > 0 and 0 < group1 < div1 and 0 < group2 < div2:
            success = np.array([group2, group1])  # successful events
            total = np.array([div2, div1])        # total number of sessions
            z_stat, p_value = sm.stats.proportions_ztest(success, total)
        else:
            z_stat, p_value = np.nan, np.nan

        # Metric change in percentage relative to the control group
        metric_change = ((conversion_2 - conversion_1) / conversion_1 * 100
            if conversion_1 > 0 else np.nan)

        # Check statistical significance (p < 0.05)
        significant = (p_value < 0.05) if not np.isnan(p_value) else False

        # Save result to list
        results.append({
            "test_number": test_number,        # test number
            "metric": event_name,              # metric name
            "numerator_event": event_name,     # numerator event
            "denominator_event": "session",    # denominator event
            "numerator_1": group1,             # number of events in group 1
            "denominator_1": div1,             # number of sessions in group 1
            "control_rate": conversion_1,      # conversion rate in group 1
            "numerator_2": group2,             # number of events in group 2
            "denominator_2": div2,             # number of sessions in group 2
            "test_rate": conversion_2,         # conversion rate in group 2
            "metric_change": metric_change,    # percentage change
            "z_stat": z_stat,                  # z-statistic
            "p_value": p_value,                # p-value
            "significant": significant         # significance (True/False)
        })

# Build the final results table
results_df = pd.DataFrame(results)
results_df

,test_number,metric,numerator_event,denominator_event,numerator_1,denominator_1,control_rate,numerator_2,denominator_2,test_rate,metric_change,z_stat,p_value,significant
0,1,add_payment_info,add_payment_info,session,1988,45362,0.043825,2229,45193,0.049322,12.542021,3.924884,0.000087,True
1,1,add_shipping_info,add_shipping_info,session,3034,45362,0.066884,3221,45193,0.071272,6.560481,2.603571,0.009226,True
2,1,begin_checkout,begin_checkout,session,3784,45362,0.083418,4021,45193,0.088974,6.660587,2.978783,0.002894,True
3,1,new account,new account,session,3823,45362,0.084278,3681,45193,0.081451,-3.354299,-1.542883,0.122859,False
4,2,add_payment_info,add_payment_info,session,2344,50637,0.046290,2409,50244,0.047946,3.576911,1.240994,0.214608,False
5,2,add_shipping_info,add_shipping_info,session,3480,50637,0.068724,3510,50244,0.069859,1.650995,0.709557,0.477979,False
6,2,begin_checkout,begin_checkout,session,4262,50637,0.084168,4313,50244,0.085841,1.988164,0.952898,0.340642,False
7,2,new account,new account,session,4165,50637,0.082252,4184,50244,0.083274,1.241934,0.588793,0.556000,False
8,3,add_payment_info,add_payment_info,session,3623,70047,0.051722,3697,70439,0.052485,1.474630,0.643172,0.520112,False
9,3,add_shipping_info,add_shipping_info,session,5298,70047,0.075635,5188,70439,0.073652,-2.621211,-1.413727,0.157442,False


In [6]:
from google.colab import files
results_df.to_csv("results_df.csv", index=False)
files.download("results_df.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 # Conclusions and Recommendations

**Test #1**

The metrics add_payment_info, add_shipping_info and begin_checkout showed statistically significant changes (p < 0.05).

However, the new account metric did not show positive results — the number of new accounts even slightly decreased.
The changes in this test affected the order placement stages but did not lead to growth in the number of new users.

**Test #2**

None of the metrics showed statistically significant changes (all p > 0.05).
The test is considered unsuccessful — the changes had no real impact on user behavior. Implementing these changes is not recommended.

**Test #3**

Only the begin_checkout metric showed a statistically significant change (p = 0.012).

Other metrics, including new account, remained without significant changes.
The test cannot be considered successful, as the changes affected only one stage of the funnel without improving the final outcome — new account creation.

**Test #4**

The metrics begin_checkout and new account showed statistically significant improvements (p < 0.05).
This is the most successful test — it positively impacted both the checkout process and new account creation.

**Recommendations**

1. Implement the changes from Test #4, as they produced statistically confirmed improvements in key funnel stages (checkout and account creation).

2. Do not implement the results of Test #2, as the changes had no effect.

3. Analyze why the improvements at the middle funnel stages in Tests #1 and #3 did not lead to growth in registrations.

## Calculation Results
https://docs.google.com/spreadsheets/d/1kawk4lMNMtAPBppOVknObBlY0HFqVt618a8DYT_pkk0/edit?gid=570772318#gid=570772318

## Tableau Dashboard
https://public.tableau.com/app/profile/iryna.savelieva/viz/ABtest_17532074898820/ABtest?publish=yes